# MLP Baseline
- supervised MLP classifier  
- A simple MLP classifier is used as a baseline model to evaluate the separability of high-dimensional sensor data under severe class imbalance.

### 구글 드라이브 마운트 및 라이브러리 설정

In [ ]:
# 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import random
import tensorflow as tf

# 재현성을 위한 시드 고정
def set_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seeds(42)

Mounted at /content/drive


### 데이터 파일 경로 설정

In [ ]:
import os

# 경로 설정
base_path = "/content/drive/MyDrive/semiconductor-yield-risk-analysis/data/"
data_file = os.path.join(base_path, "secom.data")
label_file = os.path.join(base_path, "secom_labels.data")

if os.path.exists(data_file) and os.path.exists(label_file):
    print("파일 확인 완료!")
else:
    print(f"에러: {base_path} 경로를 다시 확인해주세요.")

파일 확인 완료!


### 데이터 로딩 및 전처리

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

try:
    # 1) 데이터 로드
    X_raw = pd.read_csv(data_file, sep=r"\s+", header=None)
    y_raw = pd.read_csv(label_file, sep=r"\s+", header=None)[0].replace(-1, 0)

    # 2) 데이터 분할
    train_idx = int(len(X_raw) * 0.7)
    val_idx = int(len(X_raw) * 0.85)

    X_train_raw, y_train_raw = X_raw.iloc[:train_idx], y_raw.iloc[:train_idx]
    X_val_raw, y_val_raw = X_raw.iloc[train_idx:val_idx], y_raw.iloc[train_idx:val_idx]
    X_test_raw, y_test_raw = X_raw.iloc[val_idx:], y_raw.iloc[val_idx:]

    # 3) 전처리 함수
    def preprocess_data(X_tr, X_v, X_te):
        imputer = SimpleImputer(strategy="median")
        scaler = StandardScaler()

        # 분산 0인 센서 제거
        variances = X_tr.var()
        keep_cols = variances[variances > 1e-8].index

        X_tr_f = scaler.fit_transform(imputer.fit_transform(X_tr[keep_cols]))
        X_v_f = scaler.transform(imputer.transform(X_v[keep_cols]))
        X_te_f = scaler.transform(imputer.transform(X_te[keep_cols]))

        sensor_names = [f"Sensor_{i}" for i in keep_cols]
        return X_tr_f, X_v_f, X_te_f, sensor_names

    X_tr_s, X_v_s, X_te_s, selected_sensors = preprocess_data(X_train_raw, X_val_raw, X_test_raw)
    print(f"▶ 전처리 성공! 남은 유효 센서: {len(selected_sensors)}개")

except Exception as e:
    print(f"데이터 처리 에러: {e}")

▶ 전처리 성공! 남은 유효 센서: 468개


# [1] MLP Baseline

In [ ]:
from tensorflow.keras import layers, models

# MLP 모델 정의
def build_mlp(input_dim):
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),

        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),

        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),

        layers.Dense(64, activation='relu'),

        layers.Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )

    return model

### 학습

In [ ]:
input_dim = X_tr_s.shape[1]

mlp = build_mlp(input_dim)

history = mlp.fit(
    X_tr_s, y_train_raw,
    validation_data=(X_v_s, y_val_raw),
    epochs=30,
    batch_size=32,
    verbose=1
)

Epoch 1/30
35/35 ━━━━━━━━━━━━━━━━━━━━ 15s 158ms/step - accuracy: 0.3057 - auc: 0.5262 - loss: 1.2675 - val_accuracy: 0.9021 - val_auc: 0.4761 - val_loss: 0.4216
Epoch 2/30
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8768 - auc: 0.7724 - loss: 0.3471 - val_accuracy: 0.9277 - val_auc: 0.4552 - val_loss: 0.2960
Epoch 3/30
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9297 - auc: 0.8348 - loss: 0.2147 - val_accuracy: 0.9277 - val_auc: 0.4862 - val_loss: 0.2769
Epoch 4/30
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9425 - auc: 0.9477 - loss: 0.1560 - val_accuracy: 0.9277 - val_auc: 0.5152 - val_loss: 0.2785
Epoch 5/30
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9571 - auc: 0.9713 - loss: 0.1219 - val_accuracy: 0.9277 - val_auc: 0.5451 - val_loss: 0.2911
Epoch 6/30
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9763 - auc: 0.9804 - loss: 0.0913 - val_accuracy: 0.9277 - val_auc: 0.5518 - val_loss: 0.3130
Epoch 7/30
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step

### 평가

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred_prob = mlp.predict(X_te_s).ravel()
y_pred = (y_pred_prob > 0.5).astype(int)

print("ROC-AUC:", roc_auc_score(y_test_raw, y_pred_prob))
print(classification_report(y_test_raw, y_pred))

8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 58ms/step
ROC-AUC: 0.5266764561918748
              precision    recall  f1-score   support

           0       0.96      0.99      0.97       227
           1       0.00      0.00      0.00         9

    accuracy                           0.95       236
   macro avg       0.48      0.49      0.49       236
weighted avg       0.92      0.95      0.94       236



# [2] class imbalance 처리
- imbalance 보정
- fail(class 1)에 13배 가까운 가중치

In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

classes = np.array([0, 1])

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train_raw
)

class_weight_dict = {
    0: class_weights[0],
    1: class_weights[1]
}

print("Class weights:", class_weight_dict)

Class weights: {0: np.float64(0.5383104125736738), 1: np.float64(7.0256410256410255)}


### 학습

In [ ]:
history = mlp.fit(
    X_tr_s, y_train_raw,
    validation_data=(X_v_s, y_val_raw),
    epochs=30,
    batch_size=32,
    verbose=1,
    class_weight=class_weight_dict
)

Epoch 1/30
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9991 - auc: 1.0000 - loss: 0.0025 - val_accuracy: 0.9191 - val_auc: 0.4865 - val_loss: 0.7425
Epoch 2/30
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.9991 - auc: 1.0000 - loss: 0.0024 - val_accuracy: 0.9191 - val_auc: 0.4541 - val_loss: 0.7508
Epoch 3/30
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.9973 - auc: 1.0000 - loss: 0.0053 - val_accuracy: 0.9191 - val_auc: 0.4610 - val_loss: 0.7564
Epoch 4/30
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.9982 - auc: 1.0000 - loss: 0.0058 - val_accuracy: 0.9234 - val_auc: 0.4679 - val_loss: 0.7667
Epoch 5/30
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 1.0000 - auc: 1.0000 - loss: 0.0027 - val_accuracy: 0.9234 - val_auc: 0.4702 - val_loss: 0.7744
Epoch 6/30
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.9964 - auc: 1.0000 - loss: 0.0054 - val_accuracy: 0.9234 - val_auc: 0.4633 - val_loss: 0.7725
Epoch 7/30
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/

### 평가
- 대부분 샘플을 “0 (정상)”으로 강하게 확신
- fail 확률은 거의 0 근처에 몰림
- 일부만 0.99까지 튐 (outlier)

In [ ]:
y_pred_prob = mlp.predict(X_te_s).ravel()

print("min:", y_pred_prob.min())
print("max:", y_pred_prob.max())
print("mean:", y_pred_prob.mean())

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
min: 0.0
max: 0.9992638
mean: 0.011984849


- fail(1)을 하나도 제대로 못 맞춤
- recall = 0
- precision도 사실상 의미 없음
- 모델이 positive class를 실제로는 분리 못함

In [ ]:
from sklearn.metrics import f1_score
import numpy as np

thresholds = np.arange(0.05, 0.95, 0.01)

best_f1 = -1
best_t = 0.5

for t in thresholds:
    y_pred = (y_pred_prob > t).astype(int)
    f1 = f1_score(y_test_raw, y_pred)

    if f1 > best_f1:
        best_f1 = f1
        best_t = t

print("Best threshold:", best_t)
print("Best F1:", best_f1)

Best threshold: 0.05
Best F1: 0.0


- 대부분 sample이 0.000x 수준
- 일부만 0.01~0.004
- 거의 전부 fail 가능성 0

In [ ]:
print(np.unique(y_pred_prob[:20]))

[2.1336937e-11 1.6021431e-07 2.0575965e-07 4.1669841e-07 6.2913568e-07
 1.8158588e-06 2.6428531e-06 1.5054396e-05 1.6120015e-05 1.6773520e-05
 1.7374237e-05 1.9013101e-05 3.2081884e-05 1.4212172e-04 3.1520572e-04
 3.7910687e-04 4.9044442e-04 7.4209081e-04 3.1785169e-03 4.7143246e-03]


# [3] 결과 해석
1. supervised mlp가 fail pattern 학습 못함
  - 590 columns 의미 모름 (fail을 설명하는 feature 없음)
  - noise 많음
  - fail 샘플이 너무 적고 class 불균형 심함

2. 지도학습보다 이상 탐지 기반 접근이 더 적합할 것으로 판단됨 (Isolation Forest)